In [ ]:
# Load data

import numpy as np
import pandas as pd

SEED = 9890
TARGET = "PERCENT_PROFICIENT"
ID_COL = "ASSESSMENT_ID"

np.random.seed(SEED)

TRAINING_SET = pd.read_csv("scores_training.csv")
TEST_SET = pd.read_csv("scores_test.csv")
SCHOOLS_INFO = pd.read_csv("school_covariates.csv")
DISTRICT_INFO = pd.read_csv("district_covariates.csv")

print("TRAINING_SET:", TRAINING_SET.shape)
print("TEST_SET:", TEST_SET.shape)
print("SCHOOLS_INFO:", SCHOOLS_INFO.shape)
print("DISTRICT_INFO:", DISTRICT_INFO.shape)

print("\nTRAINING_SET columns:")
print(TRAINING_SET.columns.tolist())

print("\nTEST_SET columns:")
print(TEST_SET.columns.tolist())

print("\nSCHOOLS_INFO columns:")
print(SCHOOLS_INFO.columns.tolist())

print("\nDISTRICT_INFO columns:")
print(DISTRICT_INFO.columns.tolist())

TRAINING_SET: (144921, 6)
TEST_SET: (48307, 5)
SCHOOLS_INFO: (4754, 52)
DISTRICT_INFO: (674, 6)

TRAINING_SET columns:
['ASSESSMENT_ID', 'SCHOOL', 'SUBGROUP_NAME', 'ASSESSMENT_NAME', 'N_STUDENTS', 'PERCENT_PROFICIENT']

TEST_SET columns:
['ASSESSMENT_ID', 'SCHOOL', 'SUBGROUP_NAME', 'ASSESSMENT_NAME', 'N_STUDENTS']

SCHOOLS_INFO columns:
['SCHOOL', 'DISTRICT', 'COUNTY', 'DISTRICT_TYPE', 'REGION', 'ATTENDANCE_RATE', 'LANGUAGE_ARTS_AVERAGE_CLASS_SIZE', 'MATHEMATICS_AVERAGE_CLASS_SIZE', 'SCIENCE_AVERAGE_CLASS_SIZE', 'HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE', 'GRADE_1_AVERAGE_CLASS_SIZE', 'GRADE_2_AVERAGE_CLASS_SIZE', 'KINDERGARTEN_AVERAGE_CLASS_SIZE', 'PERCENT_FREE_LUNCH', 'PERCENT_REDUCED_LUNCH', 'NUMBER_OF_TEACHERS', 'NUMBER_OF_COUNSELORS', 'NUMBER_OF_SOCIAL_WORKERS', 'TEACHER_TURNOVER_RATE', 'PERCENT_OF_STUDENTS_SUSPENDED', 'N_PUPILS', 'FEDERAL_FUNDING_PER_PUPIL', 'LOCAL_FUNDING_PER_PUPIL', 'PRE_K', 'K', 'GRADE_01', 'GRADE_02', 'GRADE_03', 'GRADE_04', 'GRADE_05', 'GRADE_06',

In [ ]:
# Basic target and ID checks

print("Training target summary:")
print(TRAINING_SET[TARGET].describe())

print("\nTraining target missing:")
print(TRAINING_SET[TARGET].isna().sum())

print("\nTraining ASSESSMENT_ID unique?")
print(TRAINING_SET[ID_COL].is_unique)

print("\nTest ASSESSMENT_ID unique?")
print(TEST_SET[ID_COL].is_unique)

print("\nDuplicate training IDs:")
print(TRAINING_SET[ID_COL].duplicated().sum())

print("\nDuplicate test IDs:")
print(TEST_SET[ID_COL].duplicated().sum())

Training target summary:
count    144921.000000
mean         54.183521
std          26.429925
min           0.000000
25%          33.000000
50%          53.000000
75%          76.000000
max         100.000000
Name: PERCENT_PROFICIENT, dtype: float64

Training target missing:
0

Training ASSESSMENT_ID unique?
True

Test ASSESSMENT_ID unique?
True

Duplicate training IDs:
0

Duplicate test IDs:
0


In [ ]:
# Structural overlap check

train_school = set(TRAINING_SET["SCHOOL"])
test_school = set(TEST_SET["SCHOOL"])

print("Schools in both:", len(train_school & test_school)) # find minimum match
print("Only in train:", len(train_school - test_school)) # train has extra schools
print("Only in test:", len(test_school - train_school))


# Now check SCHOOL x ASSESSMENT overlap (key insight)

train_keys = set(zip(TRAINING_SET["SCHOOL"], TRAINING_SET["ASSESSMENT_NAME"]))  # number of unique combos in train
test_keys = set(zip(TEST_SET["SCHOOL"], TEST_SET["ASSESSMENT_NAME"]))

print("\nSCHOOL x ASSESSMENT overlap:")
print("In both:", len(train_keys & test_keys))
print("Total test keys:", len(test_keys))

print("\n% of test rows where school-assessment seen in train:")
test_seen = [
    (s, a) in train_keys 
    for s, a in zip(TEST_SET["SCHOOL"], TEST_SET["ASSESSMENT_NAME"])
]

print(f"{100 * np.mean(test_seen):.2f}%")

Schools in both: 4448
Only in train: 21
Only in test: 0

SCHOOL x ASSESSMENT overlap:
In both: 28956
Total test keys: 31373

% of test rows where school-assessment seen in train:
94.22%


### Structural Insight

- 100% of test schools appear in training
- ~94% of test rows have the same (SCHOOL, ASSESSMENT) seen in training

This means:

1. We are NOT predicting unseen schools
2. We often have sibling subgroup rows for the same school & assessment
3. The problem is closer to **structured missing-value completion** than pure ML

Implication:

- School-level and school-assessment-level features will be extremely strong
- Subgroup relationships (Male/Female, Econ/Not Econ) can be exploited
- Deterministic or semi-deterministic reconstruction may outperform generic models

This insight will guide feature engineering and model design.

In [ ]:
# Join covariates (simple, readable), this join is correct and has been checked

train_full = TRAINING_SET.merge(SCHOOLS_INFO, on="SCHOOL", how="left")
train_full = train_full.merge(DISTRICT_INFO, on="DISTRICT", how="left")   # we join on districts second since the earlier merge incorporated districts into the dataset

test_full = TEST_SET.merge(SCHOOLS_INFO, on="SCHOOL", how="left")
test_full = test_full.merge(DISTRICT_INFO, on="DISTRICT", how="left")

print("train_full:", train_full.shape)
print("test_full:", test_full.shape)


train_full: (144921, 62)
test_full: (48307, 61)


In [ ]:
# Missingness analysis for joined train/test tables

missing_train = pd.DataFrame({
    "train_missing_count": train_full.isna().sum(),
    "train_missing_pct": train_full.isna().mean() * 100
})

missing_test = pd.DataFrame({
    "test_missing_count": test_full.isna().sum(),
    "test_missing_pct": test_full.isna().mean() * 100
})

missing_report = missing_train.join(missing_test, how="outer")
missing_report = missing_report.fillna(0)

missing_report["max_missing_pct"] = missing_report[
    ["train_missing_pct", "test_missing_pct"]
].max(axis=1)

missing_report = missing_report.sort_values("max_missing_pct", ascending=False)

print("Columns with any missingness:")
print((missing_report["max_missing_pct"] > 0).sum())

print("\nMissingness report:")
missing_report[missing_report["max_missing_pct"] > 0]

Columns with any missingness:
52

Missingness report:


,train_missing_count,train_missing_pct,test_missing_count,test_missing_pct,max_missing_pct
TEACHER_TURNOVER_RATE,134136,92.558014,44803.0,92.746393,92.746393
KINDERGARTEN_AVERAGE_CLASS_SIZE,103467,71.395450,34487.0,71.391310,71.395450
GRADE_1_AVERAGE_CLASS_SIZE,102899,71.003512,34359.0,71.126338,71.126338
GRADE_2_AVERAGE_CLASS_SIZE,102779,70.920709,34304.0,71.012483,71.012483
HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE,89314,61.629439,29614.0,61.303745,61.629439
PERCENT_NON_DIPLOMA,16061,11.082590,5219.0,10.803817,11.082590
PERCENT_DIPLOMA,16061,11.082590,5219.0,10.803817,11.082590
PERCENT_DROPOUT,16061,11.082590,5219.0,10.803817,11.082590
PERCENT_GED,16061,11.082590,5219.0,10.803817,11.082590
PERCENT_STILL_ENROLLED,16061,11.082590,5219.0,10.803817,11.082590


### Missingness Interpretation

The joined train/test tables contain structured missingness.

Key observations:

- `TEACHER_TURNOVER_RATE` is missing for about 93% of rows, so it should not be used in the first/simple feature set.
- Early-grade class size variables (`KINDERGARTEN_AVERAGE_CLASS_SIZE`, `GRADE_1_AVERAGE_CLASS_SIZE`, `GRADE_2_AVERAGE_CLASS_SIZE`) are missing for about 71% of rows. These are likely less relevant because the assessment data mostly concerns grades 3+ and Regents-style assessments.
- `HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE` has high missingness, but may still be useful for history/Regents-related assessments.
- District graduation variables are missing for about 11% of rows. These should be kept, but later we should add missingness indicators.
- Many school-level demographic and enrollment variables are missing for about 2% of rows. The train/test missingness rates are similar, suggesting this is structured covariate missingness rather than a test-set mismatch.

Modeling implication:

We should not drop columns yet. Instead, we will keep the raw joined data intact, create missingness indicators later, and let different feature spaces decide which variables to use.

In [ ]:
# Check whether school-level missingness (2%) is concentrated in the same rows/schools

school_missing_col = "ATTENDANCE_RATE"
district_missing_col = "PERCENT_DIPLOMA"

print("School-level missingness using:", school_missing_col)

print("\nTrain rows missing school covariates:")
print(train_full[school_missing_col].isna().sum())

print("Train unique schools missing school covariates:")
print(train_full.loc[train_full[school_missing_col].isna(), "SCHOOL"].nunique())

print("\nTest rows missing school covariates:")
print(test_full[school_missing_col].isna().sum())

print("Test unique schools missing school covariates:")
print(test_full.loc[test_full[school_missing_col].isna(), "SCHOOL"].nunique())


print("\nDistrict-level missingness using:", district_missing_col)

print("\nTrain rows missing district covariates:")
print(train_full[district_missing_col].isna().sum())

print("Train unique districts missing district covariates:")
print(train_full.loc[train_full[district_missing_col].isna(), "DISTRICT"].nunique())

print("\nTest rows missing district covariates:")
print(test_full[district_missing_col].isna().sum())

print("Test unique districts missing district covariates:")
print(test_full.loc[test_full[district_missing_col].isna(), "DISTRICT"].nunique())

School-level missingness using: ATTENDANCE_RATE

Train rows missing school covariates:
3036
Train unique schools missing school covariates:
94

Test rows missing school covariates:
983
Test unique schools missing school covariates:
93

District-level missingness using: PERCENT_DIPLOMA

Train rows missing district covariates:
16061
Train unique districts missing district covariates:
38

Test rows missing district covariates:
5219
Test unique districts missing district covariates:
36


### Structured Missingness Insight

Missingness is not random.

School-level covariate missingness:
- ~3,000 rows missing
- but only ~94 schools → entire schools missing data

District-level covariate missingness:
- ~16,000 rows missing
- but only ~38 districts → entire districts missing data

Implication:

- Missingness is **group-level**, not row-level
- We should NOT:
  - drop rows
  - blindly impute means
- Instead:
  - create missingness indicators (school_missing, district_missing)
  - preserve all rows

This will be used in feature engineering later.

In [ ]:
# Basic structural features (very simple, very important)

train_full["school_missing"] = train_full["ATTENDANCE_RATE"].isna().astype(int)
test_full["school_missing"] = test_full["ATTENDANCE_RATE"].isna().astype(int)

train_full["district_missing"] = train_full["PERCENT_DIPLOMA"].isna().astype(int)
test_full["district_missing"] = test_full["PERCENT_DIPLOMA"].isna().astype(int)

print("Train school_missing distribution:")
print(train_full["school_missing"].value_counts())

print("\nTrain district_missing distribution:")
print(train_full["district_missing"].value_counts())

Train school_missing distribution:
school_missing
0    141885
1      3036
Name: count, dtype: int64

Train district_missing distribution:
district_missing
0    128860
1     16061
Name: count, dtype: int64


In [ ]:
# Missingness indicator features
# Keep only column-specific missingness indicators.
# Do NOT add broad sentinel flags that duplicate individual indicators.

# Remove missingness indicators from earlier experimental cells, if any exist
old_missing_cols_train = [col for col in train_full.columns if col.endswith("_missing")]
old_missing_cols_test = [col for col in test_full.columns if col.endswith("_missing")]

train_full = train_full.drop(columns=old_missing_cols_train, errors="ignore")
test_full = test_full.drop(columns=old_missing_cols_test, errors="ignore")

# Rebuild missingness report from the cleaned joined tables
missing_train = pd.DataFrame({
    "train_missing_count": train_full.isna().sum(),
    "train_missing_pct": train_full.isna().mean() * 100
})

missing_test = pd.DataFrame({
    "test_missing_count": test_full.isna().sum(),
    "test_missing_pct": test_full.isna().mean() * 100
})

missing_report = missing_train.join(missing_test, how="outer").fillna(0)

missing_report["max_missing_pct"] = missing_report[
    ["train_missing_pct", "test_missing_pct"]
].max(axis=1)

missing_report = missing_report.sort_values("max_missing_pct", ascending=False)

# Create column-specific indicators only
missing_cols = missing_report.index[missing_report["max_missing_pct"] > 0].tolist()
missing_cols = [col for col in missing_cols if col in train_full.columns and col in test_full.columns]

train_missing_indicators = train_full[missing_cols].isna().astype(int)
test_missing_indicators = test_full[missing_cols].isna().astype(int)

train_missing_indicators.columns = [col + "_missing" for col in missing_cols]
test_missing_indicators.columns = [col + "_missing" for col in missing_cols]

train_full = pd.concat([train_full, train_missing_indicators], axis=1)
test_full = pd.concat([test_full, test_missing_indicators], axis=1)

missing_indicator_cols = train_missing_indicators.columns.tolist()

print("Number of missingness indicator columns created:")
print(len(missing_indicator_cols))

print("\nFirst 15 missingness indicator columns:")
print(missing_indicator_cols[:15])

print("\nRedundant broad flags present?")
print([col for col in ["school_core_missing", "district_grad_missing"] if col in train_full.columns])

Broad missingness indicators:
school_core_missing       3036
district_grad_missing    16061
dtype: int64

Number of column-specific missingness indicators created:
52

First 15 missingness indicator columns:
['TEACHER_TURNOVER_RATE_missing', 'KINDERGARTEN_AVERAGE_CLASS_SIZE_missing', 'GRADE_1_AVERAGE_CLASS_SIZE_missing', 'GRADE_2_AVERAGE_CLASS_SIZE_missing', 'HISTORY_GOVERNMENT_AND_GEOGRAPHY_AVERAGE_CLASS_SIZE_missing', 'PERCENT_NON_DIPLOMA_missing', 'PERCENT_DIPLOMA_missing', 'PERCENT_DROPOUT_missing', 'PERCENT_GED_missing', 'PERCENT_STILL_ENROLLED_missing', 'SCIENCE_AVERAGE_CLASS_SIZE_missing', 'LANGUAGE_ARTS_AVERAGE_CLASS_SIZE_missing', 'MATHEMATICS_AVERAGE_CLASS_SIZE_missing', 'FEDERAL_FUNDING_PER_PUPIL_missing', 'N_PUPILS_missing']


/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_42451/2291236402.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_full[new_col] = train_full[col].isna().astype(int)
/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_42451/2291236402.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_full[new_col] = train_full[col].isna().astype(int)
/var/folders/cb/fffq6hxx2qvgkbh5yps70l_w0000gn/T/ipykernel_42451/2291236402.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result